# PySpark practice 9: joins

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-09-joins.ipynb)

28 join exercises, from everyday enrichment to interview classics and production patterns (upserts, SCD2,
as-of joins, skew). Each one names the [reference entries](pyspark-reference-09-joins.ipynb) it practises, shows its inputs and the exact
expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session and the shared `customers`, `orders`, `order_items` and `products` tables used in
[PySpark reference 9: joins](pyspark-reference-09-joins.ipynb#setup-and-sample-data). Exercises that need other data create it in
their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

## Exercises

7 easy, 14 medium, 7 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and
compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Paid orders with customer names](#exercise-1-paid-orders-with-customer-names) | Easy | [Inner join](pyspark-reference-09-joins.ipynb#inner-join) |
| 2 | [Customers who never ordered](#exercise-2-customers-who-never-ordered) | Easy | [Left anti join](pyspark-reference-09-joins.ipynb#left-anti-join) |
| 3 | [Customers with at least one return](#exercise-3-customers-with-at-least-one-return) | Easy | [Left semi join](pyspark-reference-09-joins.ipynb#left-semi-join) |
| 4 | [Orphaned order lines](#exercise-4-orphaned-order-lines) | Easy | [Left anti join](pyspark-reference-09-joins.ipynb#left-anti-join) |
| 5 | [Every customer's orders and revenue, including zeros](#exercise-5-every-customers-orders-and-revenue-including-zeros) | Easy | [Left outer join](pyspark-reference-09-joins.ipynb#left-outer-join), [Nulls introduced by outer joins](pyspark-reference-09-joins.ipynb#nulls-introduced-by-outer-joins) |
| 6 | [Calendar scaffold for daily counts](#exercise-6-calendar-scaffold-for-daily-counts) | Easy | [`DataFrame.crossJoin(other)`](pyspark-reference-09-joins.ipynb#dataframecrossjoinother) |
| 7 | [Revenue by category without losing lines](#exercise-7-revenue-by-category-without-losing-lines) | Medium | [Enrich with a lookup and a default](pyspark-reference-09-joins.ipynb#enrich-with-a-lookup-and-a-default) |
| 8 | [Paid orders per customer, keeping customers without any](#exercise-8-paid-orders-per-customer-keeping-customers-without-any) | Medium | [Filtering an outer join in WHERE vs ON](pyspark-reference-09-joins.ipynb#filtering-an-outer-join-in-where-vs-on) |
| 9 | [Match keys that may be null](#exercise-9-match-keys-that-may-be-null) | Medium | [Null-safe joins](pyspark-reference-09-joins.ipynb#null-safe-joins) |
| 10 | [Join keys with different types and dirty values](#exercise-10-join-keys-with-different-types-and-dirty-values) | Medium | [Key type mismatches](pyspark-reference-09-joins.ipynb#key-type-mismatches) |
| 11 | [Lines sold below list price](#exercise-11-lines-sold-below-list-price) | Medium | [Join on expressions and aliases](pyspark-reference-09-joins.ipynb#join-on-expressions-and-aliases) |
| 12 | [Products never sold](#exercise-12-products-never-sold) | Easy | [Left anti join](pyspark-reference-09-joins.ipynb#left-anti-join) |
| 13 | [Find the cause of a join explosion](#exercise-13-find-the-cause-of-a-join-explosion) | Medium | [Duplicate keys multiply rows](pyspark-reference-09-joins.ipynb#duplicate-keys-multiply-rows) |
| 14 | [Deduplicate a dimension before joining](#exercise-14-deduplicate-a-dimension-before-joining) | Medium | [Duplicate keys multiply rows](pyspark-reference-09-joins.ipynb#duplicate-keys-multiply-rows) |
| 15 | [Employee, manager and skip-level manager](#exercise-15-employee-manager-and-skip-level-manager) | Medium | [Self joins](pyspark-reference-09-joins.ipynb#self-joins) |
| 16 | [Customer pairs in the same region](#exercise-16-customer-pairs-in-the-same-region) | Medium | [Self joins](pyspark-reference-09-joins.ipynb#self-joins) |
| 17 | [Mutual follows](#exercise-17-mutual-follows) | Medium | [Self joins](pyspark-reference-09-joins.ipynb#self-joins) |
| 18 | [Previous order amount without window functions](#exercise-18-previous-order-amount-without-window-functions) | Medium | [Self joins](pyspark-reference-09-joins.ipynb#self-joins) |
| 19 | [Customers whose every order is settled](#exercise-19-customers-whose-every-order-is-settled) | Medium | [Left semi join](pyspark-reference-09-joins.ipynb#left-semi-join), [Left anti join](pyspark-reference-09-joins.ipynb#left-anti-join) |
| 20 | [Customers who bought Pantry plus another category](#exercise-20-customers-who-bought-pantry-plus-another-category) | Medium | [Inner join](pyspark-reference-09-joins.ipynb#inner-join), [Left semi join](pyspark-reference-09-joins.ipynb#left-semi-join) |
| 21 | [Reconcile yesterday's and today's snapshot](#exercise-21-reconcile-yesterdays-and-todays-snapshot) | Medium | [Reconcile two datasets](pyspark-reference-09-joins.ipynb#reconcile-two-datasets), [Full outer join](pyspark-reference-09-joins.ipynb#full-outer-join) |
| 22 | [Upsert a table without MERGE](#exercise-22-upsert-a-table-without-merge) | Hard | [Full outer join](pyspark-reference-09-joins.ipynb#full-outer-join), [Reconcile two datasets](pyspark-reference-09-joins.ipynb#reconcile-two-datasets) |
| 23 | [SCD2 change detection](#exercise-23-scd2-change-detection) | Hard | [Null-safe joins](pyspark-reference-09-joins.ipynb#null-safe-joins), [Point-in-time (SCD2) joins](pyspark-reference-09-joins.ipynb#point-in-time-scd2-joins) |
| 24 | [Segment at the time of the order](#exercise-24-segment-at-the-time-of-the-order) | Hard | [Point-in-time (SCD2) joins](pyspark-reference-09-joins.ipynb#point-in-time-scd2-joins) |
| 25 | [Convert orders with the rate in force (as-of join)](#exercise-25-convert-orders-with-the-rate-in-force-as-of-join) | Hard | [As-of joins](pyspark-reference-09-joins.ipynb#as-of-joins) |
| 26 | [Attribute orders to overlapping campaigns](#exercise-26-attribute-orders-to-overlapping-campaigns) | Hard | [Non-equi and range joins](pyspark-reference-09-joins.ipynb#non-equi-and-range-joins) |
| 27 | [Revenue by region and category from a star schema](#exercise-27-revenue-by-region-and-category-from-a-star-schema) | Hard | [Enrich with a lookup and a default](pyspark-reference-09-joins.ipynb#enrich-with-a-lookup-and-a-default) |
| 28 | [Salt a skewed join and prove the result unchanged](#exercise-28-salt-a-skewed-join-and-prove-the-result-unchanged) | Hard | [Salting a hot key](pyspark-reference-09-joins.ipynb#salting-a-hot-key) |

## Exercise 1: Paid orders with customer names

**Difficulty:** Easy · about 5 min · **Practises:** [Inner join](pyspark-reference-09-joins.ipynb#inner-join)

*The most common join there is: attach a dimension attribute to facts.*

Return `order_id`, `name` and `amount` for every **paid** order, ordered by `order_id`.

In [ ]:
orders.select("order_id", "customer_id", "status", "amount").show()
customers.select("customer_id", "name").show()

**Expected output**

| order_id | name | amount |
| --- | --- | --- |
| 1 | Acme Corp | 120.00 |
| 4 | Bright Retail | 260.00 |
| 7 | Cyan Traders | 310.00 |
| 10 | Echo Supplies | 74.00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = (
    orders.where(F.col("status") == "paid")
    .join(customers, "customer_id")
    .select("order_id", "name", "amount")
    .orderBy("order_id")
)
result.show()

## Exercise 2: Customers who never ordered

**Difficulty:** Easy · about 5 min · **Practises:** [Left anti join](pyspark-reference-09-joins.ipynb#left-anti-join)

*Asked in almost every interview, and used daily for churn and activation reports.*

Return `customer_id` and `name` of customers with no orders, ordered by `customer_id`.

In [ ]:
customers.select("customer_id", "name").show()
orders.select("order_id", "customer_id").show()

**Expected output**

| customer_id | name |
| --- | --- |
| 7 | Gamma Sports |
| 8 | Hotel Homeware |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = customers.join(orders, "customer_id", "left_anti").select("customer_id", "name").orderBy("customer_id")
result.show()

## Exercise 3: Customers with at least one return

**Difficulty:** Easy · about 5 min · **Practises:** [Left semi join](pyspark-reference-09-joins.ipynb#left-semi-join)

*An existence check that must not duplicate customers with several returns.*

Return `customer_id` and `name` of customers who have at least one `returned` order, each customer once, ordered by `customer_id`.

In [ ]:
orders.select("order_id", "customer_id", "status").show()

**Expected output**

| customer_id | name |
| --- | --- |
| 1 | Acme Corp |
| 4 | Delta Grocers |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
returns = orders.where(F.col("status") == "returned")
result = customers.join(returns, "customer_id", "left_semi").select("customer_id", "name").orderBy("customer_id")
result.show()

## Exercise 4: Orphaned order lines

**Difficulty:** Easy · about 5 min · **Practises:** [Left anti join](pyspark-reference-09-joins.ipynb#left-anti-join)

*A referential-integrity check that belongs in every pipeline's quality gate.*

Return `order_id` and `sku` of order lines whose SKU has no row in `products`, ordered by `order_id`, `sku`.

In [ ]:
order_items.select("order_id", "sku").show(20)
products.select("sku").show(20)

**Expected output**

| order_id | sku |
| --- | --- |
| 12 | SKU-4014 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = order_items.join(products, "sku", "left_anti").select("order_id", "sku").orderBy("order_id", "sku")
result.show()

## Exercise 5: Every customer's orders and revenue, including zeros

**Difficulty:** Easy · about 10 min · **Practises:** [Left outer join](pyspark-reference-09-joins.ipynb#left-outer-join), [Nulls introduced by outer joins](pyspark-reference-09-joins.ipynb#nulls-introduced-by-outer-joins)

*A customer report must list every customer; null revenue reads as "unknown", not "none".*

Return `customer_id`, `name`, `orders` (number of orders) and `revenue` (sum of `amount`) for **every** customer. Customers without orders show 0 and 0.00. Order by `customer_id`.

In [ ]:
customers.select("customer_id", "name").show()
orders.select("order_id", "customer_id", "amount").show()

**Expected output**

| customer_id | name | orders | revenue |
| --- | --- | --- | --- |
| 1 | Acme Corp | 3 | 234.50 |
| 2 | Bright Retail | 2 | 315.25 |
| 3 | Cyan Traders | 2 | 459.00 |
| 4 | Delta Grocers | 2 | 1269.90 |
| 5 | Echo Supplies | 1 | 74.00 |
| 6 | Foxtrot Foods | 2 | 306.40 |
| 7 | Gamma Sports | 0 | 0.00 |
| 8 | Hotel Homeware | 0 | 0.00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = (
    customers.join(orders, "customer_id", "left")
    .groupBy("customer_id", "name")
    .agg(F.count("order_id").alias("orders"), F.coalesce(F.sum("amount"), F.lit(0)).alias("revenue"))
    .orderBy("customer_id")
)
result.show()

## Exercise 6: Calendar scaffold for daily counts

**Difficulty:** Easy · about 10 min · **Practises:** [`DataFrame.crossJoin(other)`](pyspark-reference-09-joins.ipynb#dataframecrossjoinother)

*Dashboards need a row for days with no activity; a cross join builds the scaffold a left join fills in.*

For customers 1, 2 and 3 and each day from 2026-01-05 to 2026-01-07, return `customer_id`, `day` and `orders` (orders placed that UTC day, 0 when none). Order by `customer_id`, `day`.

In [ ]:
orders.where("customer_id IN (1, 2, 3)").select("order_id", "customer_id", "order_ts").show()

**Expected output**

| customer_id | day | orders |
| --- | --- | --- |
| 1 | 2026-01-05 | 3 |
| 1 | 2026-01-06 | 0 |
| 1 | 2026-01-07 | 0 |
| 2 | 2026-01-05 | 0 |
| 2 | 2026-01-06 | 1 |
| 2 | 2026-01-07 | 1 |
| 3 | 2026-01-05 | 0 |
| 3 | 2026-01-06 | 0 |
| 3 | 2026-01-07 | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
days = spark.range(3).select(F.date_add(F.lit("2026-01-05").cast("date"), F.col("id").cast("int")).alias("day"))
daily = orders.groupBy("customer_id", F.to_date("order_ts").alias("day")).agg(F.count("*").alias("orders"))
result = (
    customers.where("customer_id IN (1, 2, 3)").select("customer_id")
    .crossJoin(days)
    .join(daily, ["customer_id", "day"], "left")
    .select("customer_id", "day", F.coalesce("orders", F.lit(0)).alias("orders"))
    .orderBy("customer_id", "day")
)
result.show()

## Exercise 7: Revenue by category without losing lines

**Difficulty:** Medium · about 10 min · **Practises:** [Enrich with a lookup and a default](pyspark-reference-09-joins.ipynb#enrich-with-a-lookup-and-a-default)

*An inner join here silently drops revenue; finance will notice the total is wrong.*

Compute line revenue (`qty * unit_price`) per product `category`. Lines whose SKU has no product row count under `unknown`. Return `category` and `revenue`, ordered by `category`. The total must equal the revenue of all lines.

In [ ]:
order_items.show(20)
products.select("sku", "category").show(20)

**Expected output**

| category | revenue |
| --- | --- |
| Garden | 498.90 |
| Kitchen | 258.50 |
| Outdoor | 520.00 |
| Pantry | 1340.25 |
| unknown | 41.40 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = (
    order_items.join(products.select("sku", "category"), "sku", "left")
    .groupBy(F.coalesce("category", F.lit("unknown")).alias("category"))
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("revenue"))
    .orderBy("category")
)
result.show()

## Exercise 8: Paid orders per customer, keeping customers without any

**Difficulty:** Medium · about 10 min · **Practises:** [Filtering an outer join in WHERE vs ON](pyspark-reference-09-joins.ipynb#filtering-an-outer-join-in-where-vs-on)

*The WHERE-vs-ON trap: a filter in the wrong place turns a left join into an inner join.*

For every customer in the `North` or `East` region, return `customer_id`, `name` and `paid_orders` (number of paid orders, 0 when none). Order by `customer_id`.

In [ ]:
customers.select("customer_id", "name", "region").show()
orders.select("order_id", "customer_id", "status").show()

**Expected output**

| customer_id | name | paid_orders |
| --- | --- | --- |
| 1 | Acme Corp | 1 |
| 2 | Bright Retail | 1 |
| 7 | Gamma Sports | 0 |
| 8 | Hotel Homeware | 0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
c, o = customers.where(F.col("region").isin("North", "East")).alias("c"), orders.alias("o")
paid = (F.col("c.customer_id") == F.col("o.customer_id")) & (F.col("o.status") == "paid")
result = (
    c.join(o, paid, "left")
    .groupBy("c.customer_id", "c.name")
    .agg(F.count("o.order_id").alias("paid_orders"))
    .orderBy("customer_id")
)
result.show()

## Exercise 9: Match keys that may be null

**Difficulty:** Medium · about 10 min · **Practises:** [Null-safe joins](pyspark-reference-09-joins.ipynb#null-safe-joins)

*Composite keys with "all" rows stored as null are common in targets and budgets tables.*

Join `targets` to `actuals` on both `country` and `region`, where a null region means "the whole country" on both sides and must match. Return `country`, `region`, `target` and `actual`, ordered by `country` then `region` with nulls first.

In [ ]:
targets = spark.createDataFrame([("DE", "Berlin", 100), ("DE", None, 500), ("FR", None, 300)], "country string, region string, target int")
actuals = spark.createDataFrame([("DE", "Berlin", 90), ("DE", None, 480), ("FR", None, 310)], "country string, region string, actual int")
targets.show()
actuals.show()

**Expected output**

| country | region | target | actual |
| --- | --- | --- | --- |
| DE | null | 500 | 480 |
| DE | Berlin | 100 | 90 |
| FR | null | 300 | 310 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
t, a = targets.alias("t"), actuals.alias("a")
result = (
    t.join(a, (F.col("t.country") == F.col("a.country")) & F.col("t.region").eqNullSafe(F.col("a.region")))
    .select("t.country", "t.region", "t.target", "a.actual")
    .orderBy("country", F.col("region").asc_nulls_first())
)
result.show()

## Exercise 10: Join keys with different types and dirty values

**Difficulty:** Medium · about 10 min · **Practises:** [Key type mismatches](pyspark-reference-09-joins.ipynb#key-type-mismatches)

*CRM exports deliver ids as padded, prefixed text; an implicit cast fails the job under ANSI mode.*

`crm` holds customer ids as text such as `" 2"` and `"C-3"`. Clean them (trim, drop the `C-` prefix), cast to `int`, and return `customer_id`, `name` and `tier` for the customers that match. Order by `customer_id`.

In [ ]:
crm = spark.createDataFrame([("1", "gold"), (" 2", "silver"), ("C-3", "bronze"), ("X-9", "gold")], "customer_id string, tier string")
crm.show()

**Expected output**

| customer_id | name | tier |
| --- | --- | --- |
| 1 | Acme Corp | gold |
| 2 | Bright Retail | silver |
| 3 | Cyan Traders | bronze |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
clean = crm.withColumn("customer_id", F.regexp_replace(F.trim("customer_id"), "^C-", "").try_cast("int"))
result = customers.join(clean, "customer_id").select("customer_id", "name", "tier").orderBy("customer_id")
result.show()

## Exercise 11: Lines sold below list price

**Difficulty:** Medium · about 10 min · **Practises:** [Join on expressions and aliases](pyspark-reference-09-joins.ipynb#join-on-expressions-and-aliases)

*Margin leakage reports compare transaction prices with the catalog.*

Return `order_id`, `sku`, `unit_price`, `list_price` and `discount` (`list_price - unit_price`) for order lines sold below their product's list price. Order by `order_id`, `sku`.

In [ ]:
order_items.show(20)
products.show(20)

**Expected output**

| order_id | sku | unit_price | list_price | discount |
| --- | --- | --- | --- | --- |
| 3 | SKU-4002 | 30.00 | 34.50 | 4.50 |
| 12 | SKU-4005 | 55.00 | 55.25 | 0.25 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
i, p = order_items.alias("i"), products.alias("p")
result = (
    i.join(p, (F.col("i.sku") == F.col("p.sku")) & (F.col("i.unit_price") < F.col("p.list_price")))
    .select("i.order_id", "i.sku", "i.unit_price", "p.list_price", (F.col("p.list_price") - F.col("i.unit_price")).alias("discount"))
    .orderBy("order_id", "sku")
)
result.show()

## Exercise 12: Products never sold

**Difficulty:** Easy · about 5 min · **Practises:** [Left anti join](pyspark-reference-09-joins.ipynb#left-anti-join)

*Dead-stock and catalog clean-up reports.*

`catalog_v2` is the product catalog after two new SKUs were added. Return `sku` and `category` of catalog products that appear on no order line, ordered by `sku`.

In [ ]:
catalog_v2 = products.unionByName(spark.createDataFrame(
    [("SKU-4015", "Garden", D("19.99")), ("SKU-4016", "Kitchen", D("8.50"))], products.schema))
catalog_v2.select("sku", "category").show(20)
order_items.select("sku").distinct().orderBy("sku").show(20)

**Expected output**

| sku | category |
| --- | --- |
| SKU-4015 | Garden |
| SKU-4016 | Kitchen |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = catalog_v2.join(order_items, "sku", "left_anti").select("sku", "category").orderBy("sku")
result.show()

## Exercise 13: Find the cause of a join explosion

**Difficulty:** Medium · about 15 min · **Practises:** [Duplicate keys multiply rows](pyspark-reference-09-joins.ipynb#duplicate-keys-multiply-rows)

*"Revenue doubled after the release": a duplicated dimension key is the usual suspect.*

`catalog` lists some SKUs twice (once per supplier feed). For each duplicated SKU return `sku`, `catalog_rows`, `order_lines` (lines in `order_items` for that SKU) and `extra_rows` (rows a naive join adds: `order_lines * (catalog_rows - 1)`). Order by `sku`.

In [ ]:
catalog = products.select("sku", "category").unionByName(
    spark.createDataFrame([("SKU-4002", "Kitchen"), ("SKU-4005", "Grocery"), ("SKU-4005", "Pantry")], "sku string, category string"))
catalog.orderBy("sku").show(20)

**Expected output**

| sku | catalog_rows | order_lines | extra_rows |
| --- | --- | --- | --- |
| SKU-4002 | 2 | 2 | 2 |
| SKU-4005 | 3 | 2 | 4 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
dupes = catalog.groupBy("sku").agg(F.count("*").alias("catalog_rows")).where("catalog_rows > 1")
lines = order_items.groupBy("sku").agg(F.count("*").alias("order_lines"))
result = (
    dupes.join(lines, "sku", "left")
    .select("sku", "catalog_rows", F.coalesce("order_lines", F.lit(0)).alias("order_lines"))
    .withColumn("extra_rows", F.col("order_lines") * (F.col("catalog_rows") - 1))
    .orderBy("sku")
)
result.show()

## Exercise 14: Deduplicate a dimension before joining

**Difficulty:** Medium · about 15 min · **Practises:** [Duplicate keys multiply rows](pyspark-reference-09-joins.ipynb#duplicate-keys-multiply-rows)

*Attach the current version of a slowly changing attribute without multiplying facts.*

`regions` has several versions per customer. Keep each customer's latest version (by `updated_at`), then attach the region to every order of customers 1 to 3. Return `order_id` and `region`, ordered by `order_id`; the row count must equal the number of those orders.

In [ ]:
regions = spark.createDataFrame(
    [(1, "North", datetime(2025, 1, 1)), (1, "West", datetime(2026, 1, 1)), (2, "North", datetime(2025, 6, 1)),
     (3, "South", datetime(2025, 1, 1)), (3, "Central", datetime(2025, 3, 1))],
    "customer_id int, region string, updated_at timestamp",
)
regions.show()

**Expected output**

| order_id | region |
| --- | --- |
| 1 | West |
| 2 | West |
| 3 | West |
| 4 | North |
| 5 | North |
| 6 | Central |
| 7 | Central |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
latest = Window.partitionBy("customer_id").orderBy(F.desc("updated_at"))
current = regions.withColumn("rn", F.row_number().over(latest)).where("rn = 1").select("customer_id", "region")
result = (
    orders.where("customer_id IN (1, 2, 3)").join(current, "customer_id", "left")
    .select("order_id", "region").orderBy("order_id")
)
result.show()

## Exercise 15: Employee, manager and skip-level manager

**Difficulty:** Medium · about 10 min · **Practises:** [Self joins](pyspark-reference-09-joins.ipynb#self-joins)

*Hierarchies (org charts, category trees) are walked one level per self join.*

Return `employee`, `manager` and `skip_level` (the manager's manager) for every employee, null where a level does not exist. Order by `employee`.

In [ ]:
staff = spark.createDataFrame([(1, "Ana", None), (2, "Ben", 1), (3, "Cai", 1), (4, "Dee", 2), (5, "Eve", 4)], "emp_id int, name string, manager_id int")
staff.show()

**Expected output**

| employee | manager | skip_level |
| --- | --- | --- |
| Ana | null | null |
| Ben | Ana | null |
| Cai | Ana | null |
| Dee | Ben | Ana |
| Eve | Dee | Ben |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
e, m, s = staff.alias("e"), staff.alias("m"), staff.alias("s")
result = (
    e.join(m, F.col("e.manager_id") == F.col("m.emp_id"), "left")
    .join(s, F.col("m.manager_id") == F.col("s.emp_id"), "left")
    .select(F.col("e.name").alias("employee"), F.col("m.name").alias("manager"), F.col("s.name").alias("skip_level"))
    .orderBy("employee")
)
result.show()

## Exercise 16: Customer pairs in the same region

**Difficulty:** Medium · about 10 min · **Practises:** [Self joins](pyspark-reference-09-joins.ipynb#self-joins)

*Pair generation without mirrored duplicates is a classic interview question.*

Return every pair of different customers in the same region as `region`, `customer_a`, `customer_b` (names, with `customer_a` having the smaller `customer_id`). Each pair appears once; customers with no region are excluded. Order by `region`, `customer_a`.

In [ ]:
customers.select("customer_id", "name", "region").show()

**Expected output**

| region | customer_a | customer_b |
| --- | --- | --- |
| East | Gamma Sports | Hotel Homeware |
| North | Acme Corp | Bright Retail |
| West | Echo Supplies | Foxtrot Foods |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
a, b = customers.alias("a"), customers.alias("b")
result = (
    a.join(b, (F.col("a.region") == F.col("b.region")) & (F.col("a.customer_id") < F.col("b.customer_id")))
    .select("a.region", F.col("a.name").alias("customer_a"), F.col("b.name").alias("customer_b"))
    .orderBy("region", "customer_a")
)
result.show()

## Exercise 17: Mutual follows

**Difficulty:** Medium · about 10 min · **Practises:** [Self joins](pyspark-reference-09-joins.ipynb#self-joins)

*Social-graph interview staple: find symmetric relationships.*

`follows` has one row per follower → followee. Return each pair of users who follow **each other** once, as `user_a`, `user_b` with `user_a < user_b`. Order by `user_a`, `user_b`.

In [ ]:
follows = spark.createDataFrame([("ana", "ben"), ("ben", "ana"), ("ana", "cai"), ("cai", "dee"), ("dee", "cai"), ("eve", "ana")], "follower string, followee string")
follows.show()

**Expected output**

| user_a | user_b |
| --- | --- |
| ana | ben |
| cai | dee |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
f, g = follows.alias("f"), follows.alias("g")
result = (
    f.join(g, (F.col("f.follower") == F.col("g.followee")) & (F.col("f.followee") == F.col("g.follower")))
    .where(F.col("f.follower") < F.col("f.followee"))
    .select(F.col("f.follower").alias("user_a"), F.col("f.followee").alias("user_b"))
    .orderBy("user_a", "user_b")
)
result.show()

## Exercise 18: Previous order amount without window functions

**Difficulty:** Medium · about 15 min · **Practises:** [Self joins](pyspark-reference-09-joins.ipynb#self-joins)

*Interviewers ask for the self-join version to test join reasoning; know both.*

For each order of customers 1 to 3, return `order_id` and `prev_amount`: the amount of the same customer's previous order by `order_ts` (null for the first). Do not use window functions. Order by `order_id`.

In [ ]:
orders.where("customer_id IN (1, 2, 3)").select("order_id", "customer_id", "order_ts", "amount").show()

**Expected output**

| order_id | prev_amount |
| --- | --- |
| 1 | null |
| 2 | 120.00 |
| 3 | 84.50 |
| 4 | null |
| 5 | 260.00 |
| 6 | null |
| 7 | 149.00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
cur, prev = orders.where("customer_id IN (1, 2, 3)").alias("cur"), orders.alias("prev")
earlier = (F.col("cur.customer_id") == F.col("prev.customer_id")) & (F.col("prev.order_ts") < F.col("cur.order_ts"))
result = (
    cur.join(prev, earlier, "left")
    .groupBy("cur.order_id")
    .agg(F.max_by("prev.amount", "prev.order_ts").alias("prev_amount"))
    .orderBy("order_id")
)
result.show()

## Exercise 19: Customers whose every order is settled

**Difficulty:** Medium · about 10 min · **Practises:** [Left semi join](pyspark-reference-09-joins.ipynb#left-semi-join), [Left anti join](pyspark-reference-09-joins.ipynb#left-anti-join)

*"For all" questions are a semi join (has some) plus an anti join (has none that fail).*

Return `customer_id` and `name` of customers who have at least one order and **all** of whose orders are `paid` or `shipped`. Order by `customer_id`.

In [ ]:
orders.select("order_id", "customer_id", "status").show()

**Expected output**

| customer_id | name |
| --- | --- |
| 3 | Cyan Traders |
| 5 | Echo Supplies |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
unsettled = orders.where(~F.col("status").isin("paid", "shipped"))
result = (
    customers.join(orders, "customer_id", "left_semi")
    .join(unsettled, "customer_id", "left_anti")
    .select("customer_id", "name")
    .orderBy("customer_id")
)
result.show()

## Exercise 20: Customers who bought Pantry plus another category

**Difficulty:** Medium · about 15 min · **Practises:** [Inner join](pyspark-reference-09-joins.ipynb#inner-join), [Left semi join](pyspark-reference-09-joins.ipynb#left-semi-join)

*Cross-sell analysis joins three tables before a grouped condition.*

Return `customer_id` of customers who bought at least one `Pantry` product **and** at least one product from another known category. Lines without a product row do not count. Order by `customer_id`.

In [ ]:
order_items.show(20)
products.select("sku", "category").show(20)

**Expected output**

| customer_id |
| --- |
| 1 |
| 2 |
| 4 |
| 6 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
bought = (
    order_items.join(products, "sku")
    .join(orders.select("order_id", "customer_id"), "order_id")
    .groupBy("customer_id")
    .agg(F.count_if(F.col("category") == "Pantry").alias("pantry"), F.count_if(F.col("category") != "Pantry").alias("other"))
)
result = bought.where("pantry > 0 AND other > 0").select("customer_id").orderBy("customer_id")
result.show()

## Exercise 21: Reconcile yesterday's and today's snapshot

**Difficulty:** Medium · about 15 min · **Practises:** [Reconcile two datasets](pyspark-reference-09-joins.ipynb#reconcile-two-datasets), [Full outer join](pyspark-reference-09-joins.ipynb#full-outer-join)

*Snapshot diffs drive CDC from systems that cannot emit change events.*

Compare `yesterday` and `today` by `customer_id`. Return `customer_id` and `change` (`inserted`, `deleted` or `updated`) for every customer that changed; a tier going from null to null is not a change. Order by `customer_id`.

In [ ]:
yesterday = spark.createDataFrame([(1, "gold"), (2, "silver"), (3, None), (4, None), (5, "bronze")], "customer_id int, tier string")
today = spark.createDataFrame([(1, "gold"), (2, "gold"), (3, None), (4, "silver"), (6, "bronze")], "customer_id int, tier string")
yesterday.show()
today.show()

**Expected output**

| customer_id | change |
| --- | --- |
| 2 | updated |
| 4 | updated |
| 5 | deleted |
| 6 | inserted |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
y, t = yesterday.alias("y"), today.alias("t")
result = (
    y.join(t, F.col("y.customer_id") == F.col("t.customer_id"), "full")
    .select(
        F.coalesce("y.customer_id", "t.customer_id").alias("customer_id"),
        F.when(F.col("y.customer_id").isNull(), "inserted")
         .when(F.col("t.customer_id").isNull(), "deleted")
         .when(~F.col("y.tier").eqNullSafe(F.col("t.tier")), "updated").alias("change"),
    )
    .where(F.col("change").isNotNull())
    .orderBy("customer_id")
)
result.show()

## Exercise 22: Upsert a table without MERGE

**Difficulty:** Hard · about 20 min · **Practises:** [Full outer join](pyspark-reference-09-joins.ipynb#full-outer-join), [Reconcile two datasets](pyspark-reference-09-joins.ipynb#reconcile-two-datasets)

*Plain Parquet tables have no MERGE; jobs rebuild the target from target + updates.*

Build the new version of `target` from `updates`: a key in both takes the update **only if its `updated_at` is newer**; new keys are inserted; keys only in `target` are kept. Return `customer_id`, `tier` and `updated_at`, ordered by `customer_id`.

In [ ]:
target = spark.createDataFrame(
    [(1, "gold", datetime(2026, 1, 1)), (2, "silver", datetime(2026, 1, 1)), (3, "bronze", datetime(2026, 1, 5))],
    "customer_id int, tier string, updated_at timestamp")
updates = spark.createDataFrame(
    [(2, "gold", datetime(2026, 1, 3)), (3, "silver", datetime(2026, 1, 2)), (4, "bronze", datetime(2026, 1, 4))],
    "customer_id int, tier string, updated_at timestamp")
target.show()
updates.show()

**Expected output**

| customer_id | tier | updated_at |
| --- | --- | --- |
| 1 | gold | 2026-01-01 00:00:00 |
| 2 | gold | 2026-01-03 00:00:00 |
| 3 | bronze | 2026-01-05 00:00:00 |
| 4 | bronze | 2026-01-04 00:00:00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
t, u = target.alias("t"), updates.alias("u")
take_update = F.col("t.customer_id").isNull() | (F.col("u.updated_at") > F.col("t.updated_at"))
result = (
    t.join(u, F.col("t.customer_id") == F.col("u.customer_id"), "full")
    .select(
        F.coalesce("t.customer_id", "u.customer_id").alias("customer_id"),
        F.when(take_update & F.col("u.customer_id").isNotNull(), F.col("u.tier")).otherwise(F.col("t.tier")).alias("tier"),
        F.when(take_update & F.col("u.customer_id").isNotNull(), F.col("u.updated_at")).otherwise(F.col("t.updated_at")).alias("updated_at"),
    )
    .orderBy("customer_id")
)
result.show()

## Exercise 23: SCD2 change detection

**Difficulty:** Hard · about 20 min · **Practises:** [Null-safe joins](pyspark-reference-09-joins.ipynb#null-safe-joins), [Point-in-time (SCD2) joins](pyspark-reference-09-joins.ipynb#point-in-time-scd2-joins)

*Type 2 dimensions need to know which keys to expire and which rows to insert on every load.*

Compare the current rows of `dim` (`is_current = true`) with `incoming`. Return `customer_id` and `action`: `expire_and_insert` when the region changed (null-safe), `insert` for a new customer. Unchanged customers and customers missing from `incoming` are not returned. Order by `customer_id`.

In [ ]:
dim = spark.createDataFrame(
    [(1, "North", False), (1, "West", True), (2, "North", True), (3, None, True), (4, "East", True)],
    "customer_id int, region string, is_current boolean")
incoming = spark.createDataFrame([(1, "West"), (2, "South"), (3, None), (5, "East")], "customer_id int, region string")
dim.show()
incoming.show()

**Expected output**

| customer_id | action |
| --- | --- |
| 2 | expire_and_insert |
| 5 | insert |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
d, i = dim.where("is_current").alias("d"), incoming.alias("i")
result = (
    i.join(d, F.col("i.customer_id") == F.col("d.customer_id"), "left")
    .select(
        F.col("i.customer_id").alias("customer_id"),
        F.when(F.col("d.customer_id").isNull(), "insert")
         .when(~F.col("i.region").eqNullSafe(F.col("d.region")), "expire_and_insert").alias("action"),
    )
    .where(F.col("action").isNotNull())
    .orderBy("customer_id")
)
result.show()

## Exercise 24: Segment at the time of the order

**Difficulty:** Hard · about 20 min · **Practises:** [Point-in-time (SCD2) joins](pyspark-reference-09-joins.ipynb#point-in-time-scd2-joins)

*Point-in-time correctness: reports must use the attribute that was true when the fact happened.*

`segments` holds customer segment versions with a half-open interval `[valid_from, valid_to)`; a null `valid_to` means current. For each order of customers 1 and 2, return `order_id` and the `segment` valid at `order_ts` (null if none). Order by `order_id`.

In [ ]:
segments = spark.createDataFrame(
    [(1, "standard", datetime(2024, 1, 1), datetime(2026, 1, 5, 10, 0)), (1, "premium", datetime(2026, 1, 5, 10, 0), None),
     (2, "standard", datetime(2026, 1, 7, 0, 0), None)],
    "customer_id int, segment string, valid_from timestamp, valid_to timestamp")
orders.where("customer_id IN (1, 2)").select("order_id", "customer_id", "order_ts").show()
segments.show()

**Expected output**

| order_id | segment |
| --- | --- |
| 1 | standard |
| 2 | premium |
| 3 | premium |
| 4 | null |
| 5 | standard |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
o, s = orders.where("customer_id IN (1, 2)").alias("o"), segments.alias("s")
condition = (
    (F.col("o.customer_id") == F.col("s.customer_id"))
    & (F.col("o.order_ts") >= F.col("s.valid_from"))
    & (F.col("o.order_ts") < F.coalesce(F.col("s.valid_to"), F.lit("9999-12-31").cast("timestamp")))
)
result = o.join(s, condition, "left").select("o.order_id", "s.segment").orderBy("order_id")
result.show()

## Exercise 25: Convert orders with the rate in force (as-of join)

**Difficulty:** Hard · about 20 min · **Practises:** [As-of joins](pyspark-reference-09-joins.ipynb#as-of-joins)

*Currency conversion, price lookups and sensor calibration are all as-of joins.*

Convert each order's `amount` (EUR) to USD with the latest `fx` rate whose `rate_ts` is at or before `order_ts`. Orders before the first rate get null. Return `order_id` and `usd_amount` rounded to 2 decimals, ordered by `order_id`.

In [ ]:
fx = spark.createDataFrame(
    [(datetime(2026, 1, 6), 1.08), (datetime(2026, 1, 9), 1.10), (datetime(2026, 1, 12, 12), 1.09)],
    "rate_ts timestamp, usd_rate double")
fx.show()
orders.select("order_id", "order_ts", "amount").show()

**Expected output**

| order_id | usd_amount |
| --- | --- |
| 1 | null |
| 2 | null |
| 3 | null |
| 4 | 280.8 |
| 5 | 59.67 |
| 6 | 160.92 |
| 7 | 341.0 |
| 8 | 98.89 |
| 9 | 1298.0 |
| 10 | 80.66 |
| 11 | 228.9 |
| 12 | 105.08 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
rates = fx.withColumn("valid_to", F.lead("rate_ts").over(Window.orderBy("rate_ts"))).alias("r")
o = orders.alias("o")
in_force = (F.col("o.order_ts") >= F.col("r.rate_ts")) & (
    F.col("o.order_ts") < F.coalesce(F.col("r.valid_to"), F.lit("9999-12-31").cast("timestamp")))
result = (
    o.join(F.broadcast(rates), in_force, "left")
    .select("o.order_id", F.round(F.col("o.amount") * F.col("r.usd_rate"), 2).alias("usd_amount"))
    .orderBy("order_id")
)
result.show()

## Exercise 26: Attribute orders to overlapping campaigns

**Difficulty:** Hard · about 20 min · **Practises:** [Non-equi and range joins](pyspark-reference-09-joins.ipynb#non-equi-and-range-joins)

*Marketing attribution: date-range joins with a tie-break rule when ranges overlap.*

`campaigns` have inclusive `start_date`/`end_date` ranges that can overlap. Assign each order to the active campaign (by the order's UTC date) with the **highest** `priority`, or `none` when no campaign is active. Return `order_id` and `campaign`, ordered by `order_id`.

In [ ]:
campaigns = spark.createDataFrame(
    [("new_year", date(2026, 1, 1), date(2026, 1, 7), 1), ("flash", date(2026, 1, 6), date(2026, 1, 6), 5),
     ("winter", date(2026, 1, 9), date(2026, 1, 12), 2)],
    "campaign string, start_date date, end_date date, priority int")
campaigns.show()
orders.select("order_id", "order_ts").show()

**Expected output**

| order_id | campaign |
| --- | --- |
| 1 | new_year |
| 2 | new_year |
| 3 | new_year |
| 4 | flash |
| 5 | new_year |
| 6 | new_year |
| 7 | winter |
| 8 | winter |
| 9 | winter |
| 10 | winter |
| 11 | none |
| 12 | none |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
o, c = orders.withColumn("day", F.to_date("order_ts")).alias("o"), campaigns.alias("c")
matches = o.join(F.broadcast(c), F.col("o.day").between(F.col("c.start_date"), F.col("c.end_date")), "left")
best = Window.partitionBy("o.order_id").orderBy(F.desc_nulls_last("c.priority"))
result = (
    matches.withColumn("rn", F.row_number().over(best)).where("rn = 1")
    .select("o.order_id", F.coalesce("c.campaign", F.lit("none")).alias("campaign"))
    .orderBy("order_id")
)
result.show()

## Exercise 27: Revenue by region and category from a star schema

**Difficulty:** Hard · about 15 min · **Practises:** [Enrich with a lookup and a default](pyspark-reference-09-joins.ipynb#enrich-with-a-lookup-and-a-default)

*Star-schema reports join one fact to several dimensions without losing or duplicating facts.*

Compute line revenue (`qty * unit_price`) by customer `region` and product `category`, using `unknown` where either is missing. Return `region`, `category` and `revenue`, ordered by `region`, `category`.

In [ ]:
order_items.show(20)

**Expected output**

| region | category | revenue |
| --- | --- | --- |
| North | Garden | 260.00 |
| North | Kitchen | 184.50 |
| North | Pantry | 105.25 |
| South | Garden | 149.00 |
| South | Outdoor | 310.00 |
| West | Kitchen | 74.00 |
| West | Outdoor | 210.00 |
| West | Pantry | 55.00 |
| West | unknown | 41.40 |
| unknown | Garden | 89.90 |
| unknown | Pantry | 1180.00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = (
    order_items
    .join(orders.select("order_id", "customer_id"), "order_id")
    .join(customers.select("customer_id", "region"), "customer_id", "left")
    .join(F.broadcast(products.select("sku", "category")), "sku", "left")
    .groupBy(F.coalesce("region", F.lit("unknown")).alias("region"), F.coalesce("category", F.lit("unknown")).alias("category"))
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("revenue"))
    .orderBy("region", "category")
)
result.show()

## Exercise 28: Salt a skewed join and prove the result unchanged

**Difficulty:** Hard · about 15 min · **Practises:** [Salting a hot key](pyspark-reference-09-joins.ipynb#salting-a-hot-key)

*Skew is the top cause of one straggling task; salting is the manual fix interviewers expect you to know.*

`clicks` is heavily skewed toward customer 1. Join it to `customers` with a salt of 4 buckets (salt the clicks, replicate customers once per salt) and return `name` and `clicks` (count per customer), ordered by `name`. The result must equal the unsalted join.

In [ ]:
clicks = spark.range(1000).select(F.when(F.col("id") < 900, 1).otherwise(F.col("id") % 3 + 2).alias("customer_id"), "id")
clicks.groupBy("customer_id").count().orderBy("customer_id").show()

**Expected output**

| name | clicks |
| --- | --- |
| Acme Corp | 900 |
| Bright Retail | 34 |
| Cyan Traders | 33 |
| Delta Grocers | 33 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
N = 4
salted_clicks = clicks.withColumn("salt", (F.col("id") % N).cast("int"))
salted_customers = customers.withColumn("salt", F.explode(F.sequence(F.lit(0), F.lit(N - 1))))
result = (
    salted_clicks.join(salted_customers, ["customer_id", "salt"])
    .groupBy("name").agg(F.count("*").alias("clicks"))
    .orderBy("name")
)
result.show()

**Next:** look anything up in [PySpark reference 9: joins](pyspark-reference-09-joins.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).